## Data Quality

O código apresentado neste notebook foi desenvolvido durante a análise da qualidade da base de dados e tem como objetivo compreender possíveis regras de negócio e como elas estão representadas na base de dados, além de identificar inconsistências, problemas de tipagem, valores ausentes, duplicatas e outros possíveis problemas de qualidade.

In [5]:
import pandas as pd

from pathlib import Path

In [6]:
df = pd.read_csv("../dataset/dataset_limpo_univariado.csv")

### Dados pessoais

Há alguns fatores principais que podem ser usados como referência para uma investigação de justiça algorítmica, na nossa base de dados eles são: 

- Idade
- Grau de instrução escolar
- Status civil
- Status econômico (tendo como proxy residência)
- Sexo biológico 

Abaixo vamos explorar de forma multivariada se possuímos dados suficientes para investigarmos situações de (in)justiça algorítimica com base nesses tópicos, se há alguma inconsistência e explorarmos possíveis padrões nos dados. 

In [7]:
COLUNAS_ESCOLARIDADE = [
    "QT_CLI_ANF",
    "QT_CLI_PRI_GR",
    "QT_CLI_SEG_GR",
    "QT_CLI_SUP"
]

COLUNAS_GENERO = [
    'QT_CLI_MLH', 
    'QT_CLI_HOM'
]

COLUNAS_STATUS_CIVIL = [
    "QT_CLI_STR",
    "QT_CLI_CAS",
    "QT_CLI_SPD"
]

COLUNAS_RESIDENCIAS = [
    "QT_RES_PRP",
    "QT_RES_ALU",
    "QT_RES_OUT"
]

- Análise por gênero e quantidade total de clientes

In [8]:
soma_genero = df[COLUNAS_GENERO].fillna(0).sum(axis=1) # Calcula a soma de sexo preenchendo nulos com 0

registros_consistentes = (soma_genero == df['QT_CLI']).sum()
registros_inconsistentes = (soma_genero != df['QT_CLI']).sum()
total = len(df)

print(f"Total de registros: {total}")
print(f"Registros consistentes: {registros_consistentes} ({(registros_consistentes/total)*100:.2f}%)")
print(f"Registros Inconsistentes: {registros_inconsistentes} ({(registros_inconsistentes/total)*100:.2f}%)")

Total de registros: 143168
Registros consistentes: 143168 (100.00%)
Registros Inconsistentes: 0 (0.00%)


Quis investigar se os dados se mostram consistentes no que diz respeito a quantidades registradas. Inicialmente vemos que as colunas que dizem respeito a quantidade de homens e quantidade de mulheres no grupo é compatível com a quantidade total de clientes que está registrada no grupo em 100% da base de dados. 

- Análise por nivel de escolaridade e quantidade total de clientes

In [9]:
soma_dados_escolaridade = df[COLUNAS_ESCOLARIDADE].fillna(0).sum(axis=1)

registros_consistentes = (soma_dados_escolaridade == df['QT_CLI']).sum()
registros_inconsistentes = (soma_dados_escolaridade < df['QT_CLI']).sum()
total = len(df)

print(f"Total de registros: {total}")
print(f"Registros consistentes: {registros_consistentes} ({(registros_consistentes/total)*100:.2f}%)")
print(f"Registros Inconsistentes: {registros_inconsistentes} ({(registros_inconsistentes/total)*100:.2f}%)")

Total de registros: 143168
Registros consistentes: 29 (0.02%)
Registros Inconsistentes: 143139 (99.98%)


Outra feature necessária de se investigar sua consistência é a de escolaridade dos clientes. Em 99.98% da base de dados a quantidade de clientes que indicou seu nível de escolaridade é menor que a quantidade de clientes, tornando assim, **uma feature fragil para uso como parâmetro de investigação de justiça algorítmica, sendo necessário possivelmente seu descarte.**

- Análise por status civil e quantidade de clientes

In [10]:
soma_status_civil = df[COLUNAS_STATUS_CIVIL].fillna(0).sum(axis=1)

registros_consistentes = (soma_status_civil == df['QT_CLI']).sum()
registros_inconsistentes = (soma_status_civil < df['QT_CLI']).sum()
total = len(df)

print(f"Total de registros: {total}")
print(f"Registros consistentes: {registros_consistentes} ({(registros_consistentes/total)*100:.2f}%)")
print(f"Registros Inconsistentes: {registros_inconsistentes} ({(registros_inconsistentes/total)*100:.2f}%)")

Total de registros: 143168
Registros consistentes: 140006 (97.79%)
Registros Inconsistentes: 3162 (2.21%)


No que diz respeito a informações sobre status civil, temos que há 97.8% da base com registros consistentes, ou seja, que todos os clintes do grupo indicaram seu status civil. Acredito que o melhor caminho seria manter apenas os registros consistentes. Isso é feito na célula abaixo:

In [11]:
registros_consistentes = soma_status_civil == df['QT_CLI']
df = df[registros_consistentes].reset_index(drop=True)

Por fim, verificamos a qualidade de dados referente as colunas de residências. Essas colunas indicam, a respeito da moradia dos clientes, quantas residências são próprias, quantas residências são alugadas, quantas residências são emprestadas ou mora com parentescos.

- Analise por quantidade de residências e quantidade de clientes

In [12]:
soma_residencias = df[COLUNAS_RESIDENCIAS].fillna(0).sum(axis=1)

registros_um_pra_um = (soma_residencias == df['QT_CLI']).sum() # Supõe-se que cada cliente está relacionado a uma, e apenas uma, moradia
registros_maior_que_um = (soma_residencias > df['QT_CLI']).sum() # Há mais de uma moradia por cliente do grupo
registros_menor_que_um = (soma_residencias < df['QT_CLI']).sum() # Há menos de uma moradia por cliente de dentro do grupo
total = len(df)

print(f"Total de registros: {total}")
print(f"Registros uma moradia por cliente do grupo: {registros_um_pra_um} ({(registros_um_pra_um/total)*100:.2f}%)")
print(f"Registros mais de uma moradia por cliente do grupo {registros_maior_que_um} ({(registros_maior_que_um/total)*100:.2f}%)")
print(f"Registros menos de uma moradia por cliente de dentro do grupo: {registros_menor_que_um} ({(registros_menor_que_um/total)*100:.2f}%)")

Total de registros: 140006
Registros uma moradia por cliente do grupo: 126659 (90.47%)
Registros mais de uma moradia por cliente do grupo 0 (0.00%)
Registros menos de uma moradia por cliente de dentro do grupo: 13347 (9.53%)


Apesar de não haver grupos de clientes com mais de uma moradia por pessoa, há cerca de 10% da base de dados com menos de uma moradia por cliente. Visto que as opções foram (1) residência própria, (2) residência alugada e (3) residência emprestada/mora com parentescos, o que isso quer dizer? que há clientes que não possuem moradia/moram na rua? Não faz sentido, só pode ter sido erro de registro/coleta de dados. Vamos remover esses registros momentaneamente.

In [13]:
registros_consistentes = soma_residencias == df['QT_CLI']
df = df[registros_consistentes].reset_index(drop=True)

Após a remoção ficamos com um total de 126 mil 659 registros.

In [14]:
filepath = Path("../dataset/dataset_limpo_multivariado.csv")
df.to_csv(filepath)